In [ ]:
!pip install visionist_client

In [ ]:
from visionist_client import Visionist
from IPython.display import Video, Image

import pathlib

yolo = Visionist("localhost:9068")

res = yolo.run(
    data   = {"video": pathlib.Path("../cozinha.mp4")},
    config = {"yolo": {"command": "detect",
                       "session_id": "demo-1",
                       "parameters": {"frame_step":30,
                                      "save_annotated": True}}})

print(res.fields["detections"][10])
Image(res.fields["annotated"][10])

In [ ]:
from visionist_client import Visionist
from IPython.display import Video, Image

import pathlib

tapnext = Visionist("localhost:9063")

res = tapnext.run(
    data   = {"video": pathlib.Path("../cozinha.mp4")},
    config={"tapnext": {"command": "track",
                          "parameters": {"grid_size": 32},
                          "session_id": "demo-2"}})

res

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(res.fields["observation_matrix"])


In [ ]:
from visionist_client import Visionist
import pathlib
import matplotlib.pyplot as plt

lightglue = Visionist("localhost:9069")   # box-lightglue, mapped 9069 -> 8061

img0 = pathlib.Path("../images/lightglue_box/test/00.jpg")
img1 = pathlib.Path("../images/lightglue_box/test/01.jpg")

res = lightglue.run(
    data   = {"images": [img0, img1]},
    config = {"lightglue": {"parameters": {
        "feature_extractor": "SUPERPOINT",   # or "DISK"
        "max_keypoints": 1024}}})

print(res.config)   # {"lightglue": {"status": "done", "matcher": "LightGlue (superpoint)", …}}

# fields are declared "numpy" — boxes_client hands them back as ndarrays
kp = res.keypoints            # (2, N, 2)
M  = res.matches              # (K, 2) indices — the network's own output
p0 = kp[0][M[:, 0]]           # matched points, image 0
p1 = kp[1][M[:, 1]]           # matched points, image 1
print(f"matches: {len(M)} | mean confidence: {res.confidence.mean():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, path, pts in zip(axes, (img0, img1), (p0, p1)):
    ax.imshow(plt.imread(str(path)))
    ax.scatter(pts[:, 0], pts[:, 1], s=1, c="red")
    ax.set_title(f"{path.name}: {len(pts)} matched points")
plt.tight_layout()
plt.show()
